# 01 Exploratory data analysis

SterileGuard AI, CPE 595 Group 8. Simulated Dental Central Sterile days from the calibrated SimPy model (SG-09). Synthetic data only: no patient or employee data. Set `SG_ML_DATASET` to a dataset id or parquet path (default: the newest dataset in `data/sim/`). Figures are written as 300 dpi PNGs to `packages/ml/reports/figures/`.

In [ ]:
import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sg_ml.data import ROOT

FIG_DIR = ROOT / "packages" / "ml" / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})


def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=300)
    plt.show()
    plt.close(fig)

In [ ]:
from pathlib import Path

from sg_ml.data import config_for, load_dataset
from sg_ml.features import TARGET, FeatureContext, add_features, sequence_ids

df, meta = load_dataset(os.environ.get("SG_ML_DATASET", "latest"))
ctx = FeatureContext.from_config(config_for(meta))
F = add_features(df, ctx)
F["sequence"] = sequence_ids(F)
print(meta.get("id"), len(F), "days,", F["sequence"].nunique(), "sequences")
F[
    [TARGET, "regular_patients", "morning_staff_present", "staff_hours_before_15", "capacity_ratio"]
].describe().round(2)

## Input distributions

In [ ]:
inputs = [
    "regular_patients",
    "mf_cases",
    "student_clinic_sessions",
    "morning_staff_present",
    "aft_11_staff",
    "aft_13_staff",
    "aft_15_staff",
    "outside_support_staff",
    "ot_available_staff",
    "washers_up",
    "sterilizers_up",
    "carryover_backlog_estimate",
]
fig, axes = plt.subplots(3, 4, figsize=(14, 8))
for ax, col in zip(axes.ravel(), inputs, strict=True):
    vals = pd.to_numeric(F[col])
    bins = np.arange(vals.min(), vals.max() + 2) - 0.5 if vals.nunique() < 15 else 30
    ax.hist(vals, bins=bins, color="#2b7a78")
    ax.set_title(col, fontsize=10)
save(fig, "eda_input_distributions")

## Overtime labor-hours

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(F[TARGET], bins=50, color="#3a506b")
ax.set_xlabel("Overtime labor-hours (morning operation)")
ax.set_ylabel("Days")
ax.set_title(f"Overtime per day ({(F[TARGET] == 0).mean():.0%} of days have none)")
save(fig, "eda_overtime_hist")

In [ ]:
order = ["mon", "tue", "wed", "thu", "fri"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([F.loc[F["day_of_week"] == d, TARGET] for d in order], tick_labels=order)
ax.set_ylabel("Overtime labor-hours")
ax.set_title("Overtime by weekday")
save(fig, "eda_overtime_by_weekday")

## Staffing vs overtime

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sc = axes[0].scatter(
    F["staff_hours_before_15"], F[TARGET], c=F["demand_units_est"], s=6, cmap="viridis"
)
fig.colorbar(sc, ax=axes[0], label="Expected work units")
axes[0].set_xlabel("Staff hours before 3:00 PM")
axes[0].set_ylabel("Overtime labor-hours")
axes[1].scatter(F["capacity_ratio"], F[TARGET], s=6, color="#5c4d7d")
axes[1].set_xlabel("Work units per staff hour before 3:00 PM")
save(fig, "eda_staffing_vs_overtime")

## Correlations

In [ ]:
cols = [
    *inputs,
    "staff_hours_before_15",
    "demand_units_est",
    "capacity_ratio",
    "patients_per_morning_staff",
    "ot_hours_prev_day",
    TARGET,
]
corr = F[cols].apply(pd.to_numeric).corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols)), cols, rotation=90, fontsize=8)
ax.set_yticks(range(len(cols)), cols, fontsize=8)
fig.colorbar(im, ax=ax)
ax.set_title("Pearson correlation")
save(fig, "eda_correlation_heatmap")
corr[TARGET].drop(TARGET).sort_values().round(2)

## Carryover dynamics over a sequence

Each simulated sequence is 20 linked workdays: unfinished work at the end of a day is the next day's backlog.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for s in sorted(F["sequence"].unique())[:3]:
    g = F[F["sequence"] == s].sort_values("date")
    ax.plot(range(1, len(g) + 1), g[TARGET], marker="o", label=f"sequence {s}")
ax.set_xlabel("Workday in sequence")
ax.set_ylabel("Overtime labor-hours")
ax.legend()
save(fig, "eda_carryover_sequences")
F.groupby("carryover_backlog_estimate")[TARGET].agg(["count", "mean"]).round(2)

## Provisional burden classes

Terciles of simulated overtime (thresholds_source = simulated_provisional). Final cutoffs come from real data (SG-29).

In [ ]:
t1, t2 = np.quantile(F[TARGET], [1 / 3, 2 / 3])
labels = np.where(F[TARGET] <= t1, "low", np.where(F[TARGET] <= t2, "moderate", "high"))
counts = pd.Series(labels).value_counts().reindex(["low", "moderate", "high"]).fillna(0)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(counts.index, counts.values, color=["#3aa17e", "#e0a526", "#c8553d"])
ax.set_title(f"Class balance (low <= {t1:.1f} h < moderate <= {t2:.1f} h < high)")
save(fig, "eda_burden_balance")
counts

## Simulated vs real

Real rows never enter git. When a private parquet of real days is available locally, set `SG_ML_REAL_PARQUET` to overlay it; otherwise this cell only notes that none is present.

In [ ]:
real_path = os.environ.get("SG_ML_REAL_PARQUET")
if real_path and Path(real_path).exists():
    real = pd.read_parquet(real_path)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, col in zip(axes, [TARGET, "regular_patients"], strict=True):
        ax.hist(F[col], bins=30, density=True, alpha=0.6, label="simulated")
        ax.hist(real[col], bins=10, density=True, alpha=0.6, label=f"real (n={len(real)})")
        ax.set_title(col)
        ax.legend()
    save(fig, "eda_sim_vs_real")
else:
    print("No real days in this environment: overlay skipped.")